# Resistor/current mode — offline acceptance

These tests send no commands to live hardware. Every test and assertion is visible in a cell. A temporary local settings file and a protocol double exercise the real AD5780/DACArray conversion paths and dashboard command queue. Current is **mA**, resistance is **ohms**, and ramp speed is **mA/s**. No firmware changes are involved.

Run all cells. Failures raise assertions in the notebook. Existing voltage APIs retain their voltage units. Resistor settings follow the physical address/local channel across global channel reorderings.

In [ ]:
import math, json, tempfile, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from malab.instruments.ad5780 import AD5780, ChannelCalibration, RampStatus
from malab.instruments.dac_array import DACArray
from malab.instruments.resistive_load import ResistorStore
from malab.dashboard.controller import Dashboard

temporary=tempfile.TemporaryDirectory()
base=Path(temporary.name)
def store(name):return ResistorStore(base/(name+'.local.json'))
def rejects(function, exception=ValueError):
    try:function()
    except exception:return
    raise AssertionError('Expected '+exception.__name__)

class OfflineDAC(AD5780):
    def __init__(self,**kwargs):
        kwargs['enabled']=False;kwargs['use_calibration']=False
        super().__init__(**kwargs)
        self.calls=[];self.codes=[131072]*4
    def connect(self):self.socket=True;self.calls.append('connect')
    def disconnect(self):self.socket=None
    def close(self):self.disconnect()
    def initialize(self,channel=None):self.calls.append(('INIT',channel))
    def query(self,command,timeout=None):
        self.calls.append(command);parts=command.split();name=parts[0]
        if name=='SET':
            ch,code=map(int,parts[1:]);self.codes[ch-1]=code
            return f'SET command received with parameters: {ch} {code}'
        if name=='READ':return str(self.codes[int(parts[1])-1])
        if name=='RAMP':return 'RAMP ACCEPTED '+' '.join(parts[1:])
        if name=='STATUS':return 'RAMP IDLE 0 0'
        if name=='VALUES':return 'VALUES '+' '.join(f'{v} 1' for v in self.codes)
        raise AssertionError('Unexpected mock command: '+command)

print('Ready: temporary settings and offline protocol double; no live connections.')

In [ ]:
settings=store('validation');dac=OfflineDAC(address='192.0.2.10',resistor_store=settings)
for bad in [0,-1,True,False,None,'1000',float('nan'),float('inf'),1e-320]:
    rejects(lambda bad=bad:dac.add_resistor(1,bad))
for channel in [0,5,True,1.5]:rejects(lambda channel=channel:dac.add_resistor(channel,1000))
rejects(lambda:dac.get_current(1));rejects(lambda:dac.set_current(1,1));rejects(lambda:dac.ramp_current(1,1,.1))
assert dac.calls==[]
assert dac.add_resistor(1,2000)==2000
assert dac.get_resistance(1)==2000
assert dac.calls==[], 'Metadata changes must never initialize/set an output'
print('PASS: resistance/current validation precedes hardware access; saving is metadata only.')

In [ ]:
for current in [1.25,-1.25,0]:
    actual=dac.set_current(1,current)
    assert abs(actual-current)<.00004
    assert abs(dac.get_current(1)-current)<.00004
    assert abs(dac.get_voltage(1)-current*2)<.00008
assert dac.get_current_limits(1)==(-5,9.999923706054688/2)
# Explicit voltage API keeps its meaning even when a resistor exists.
assert abs(dac.set_voltage(1,1)-1)<.00008
assert abs(dac.get_current(1)-.5)<.00004
print('PASS: signed mA set/get, voltage compatibility and calibrated limits.')

In [ ]:
dac.calls.clear()
status=dac.ramp_current(1,2.5,.05,step_size=10,wait=False)
assert status.state=='ACCEPTED'
wire=dac.calls[-1].split()
assert wire[0]=='RAMP' and int(wire[1])==1
assert int(wire[2])==dac.voltage_to_code(1,5)
expected_delay=math.ceil(10*(20/2**18)/.1*1e6)
assert int(wire[4])==expected_delay,(wire,expected_delay)
for current,speed in [(6,.1),(float('nan'),.1),(1,0),(1,-1),(1,float('inf'))]:
    dac.calls.clear();rejects(lambda:dac.ramp_current(1,current,speed,wait=False));assert dac.calls==[]
dac.calls.clear();rejects(lambda:dac.set_current(1,6));assert dac.calls==[]
# A tighter measured voltage range must also reject before SET.
dac.get_calibration=lambda channel:ChannelCalibration(-9992845256,76283553,130996,True)
rejects(lambda:dac.set_current(1,-5));assert dac.calls==[]
print('PASS: mA/s maps to device microseconds; overrange/nonfinite commands send no output writes.')

In [ ]:
settings=store('mapping')
a=DACArray(['192.0.2.10','192.0.2.20'],dac_factory=OfflineDAC,resistor_store=settings)
a.add_resistor(2,4700)
b=DACArray(['192.0.2.20','192.0.2.10'],dac_factory=OfflineDAC,resistor_store=ResistorStore(settings.path))
assert b.get_resistance(6)==4700 and b.get_resistance(2) is None
single=OfflineDAC(address='192.0.2.10:23',resistor_store=ResistorStore(settings.path))
assert single.get_resistance(2)==4700
assert abs(b.set_current(6,.5)-.5)<.00002
assert not b.devices[0].calls
assert b.devices[1].calls[-1].startswith('SET 2 ')
b.remove_resistor(6)
assert single.get_resistance(2) is None and a.get_resistance(2) is None
print('PASS: persistence, single/array API compatibility and physical-channel identity after reorder.')

In [ ]:
settings=store('concurrent')
def save_channel(channel):
    ResistorStore(settings.path).set('192.0.2.10',channel,channel*1000)
with ThreadPoolExecutor(max_workers=4) as pool:list(pool.map(save_channel,range(1,5)))
assert settings.snapshot()['192.0.2.10']=={str(i):i*1000 for i in range(1,5)}
bad=store('bad-file');bad.path.write_text('{"version":1,"units":{"192.0.2.10":{"1":0}}}',encoding='utf-8')
rejects(bad.snapshot)
assert not list(base.glob('resistors-*.local.json'))
print('PASS: independent writers preserve other channels; corrupt settings fail visibly; atomic files cleaned.')

In [ ]:
settings=store('dashboard')
d=Dashboard(['192.0.2.10','192.0.2.20'],meter_address='192.0.2.71',meter_channel=6,dac_factory=OfflineDAC,resistor_store=settings)
for unit in d.units:unit.poll_dac();unit.dac.calls.clear()
save=dict(action='resistor',unit=2,channel=6,resistance_ohms=2000,request_id='save')
d.submit(save);d.submit(save)
assert all(not u.dac.calls and u.commands.empty() for u in d.units)
assert d.snapshot()['channels'][5]['resistance_ohms']==2000
rejects(lambda:d.submit(dict(save,resistance_ohms=1000)))
request=dict(action='set_current',unit=2,channel=6,current_mA=.5,resistance_ohms=2000,request_id='set')
d.submit(request);d.submit(request)
assert d.units[1].commands.qsize()==1
queued=d.units[1].commands.get_nowait()
assert queued['action']=='set' and queued['channel']==2 and queued['voltage']==1
u=d.units[1];u.action(queued);u.complete(queued);d.submit(request)
assert u.commands.empty() and not d.units[0].dac.calls
assert u.dac.calls==[('INIT',2),'SET 2 144179'],u.dac.calls
print('PASS: dashboard current routes once to correct unit/local channel; resistor Save causes no output change.')

In [ ]:
request=dict(action='ramp_current',unit=2,channel=6,current_mA=1,speed_mA_s=.05,resistance_ohms=2000,request_id='ramp')
d.submit(request);queued=u.commands.get_nowait()
assert queued['voltage']==2 and queued['speed']==.1
u.dac.calls.clear();u.action(queued);u.complete(queued)
assert u.dac.calls[0]==('INIT',2) and u.dac.calls[-1].startswith('RAMP 2 ')
rejects(lambda:d.submit(dict(action='resistor',unit=2,channel=6,resistance_ohms=3000,request_id='busy')))
u.state['ramp']['state']='IDLE'
# A stale browser must not send voltage units or a current calculated with old R.
rejects(lambda:d.submit(dict(action='set',channel=6,voltage=1,request_id='old-voltage')))
rejects(lambda:d.submit(dict(action='set_current',channel=6,current_mA=1,resistance_ohms=1000,request_id='old-R')))
rejects(lambda:d.submit(dict(action='set_current',unit=1,channel=6,current_mA=1,resistance_ohms=2000,request_id='wrong-unit')))
# Another Python client changes R after acceptance: reject before INIT/SET.
u.dac.calls.clear();d.submit(dict(action='set_current',channel=6,current_mA=1,resistance_ohms=2000,request_id='queued'))
queued=u.commands.get_nowait();settings.set('192.0.2.20',2,3000)
rejects(lambda:u.action(queued));assert u.dac.calls==[]
u.complete(queued,'Resistance changed')
# Existing network safety remains in effect for current commands.
u.state['network']['state']='disconnected'
rejects(lambda:d.submit(dict(action='set_current',channel=6,current_mA=1,resistance_ohms=3000,request_id='offline')))
print('PASS: mA ramp conversion, busy guard, stale-mode/resistance rejection and network safety.')

In [ ]:
settings.set('192.0.2.20',2,1000)
u.state['network'].update(state='good',last_ok=time.time())
u.state['channels'][1].update(voltage=1.5,at=time.time())
d.meter.update(state='good',voltage=1.51,at=time.time())
s=d.snapshot()
assert s['channels'][5]['current_mA']==1.5
assert s['meter']['current_mA']==1.51 and s['meter']['voltage']==1.51
u.state['channels'][1]['voltage']=None
assert d.snapshot()['channels'][5]['current_mA'] is None
u.state['channels'][1]['voltage']=1.5
# One normal history sample preserves its original R and mA values.
d.stop.wait=lambda seconds:d.stop.set()
d.history_loop();h=d.snapshot()['history'][-1]
assert h['currents'][5]==1.5 and h['meter_current']==1.51 and h['resistances'][5]==1000
settings.set('192.0.2.20',2,2000)
assert d.snapshot()['history'][-1]['currents'][5]==1.5
assert d.snapshot()['channels'][5]['current_mA']==.75
print('PASS: separate meter-derived current, unknown-value propagation and history not reinterpreted after R changes.')

In [ ]:
u.dac.calls.clear()
d.submit(dict(action='resistor',unit=2,channel=6,resistance_ohms=None,request_id='remove'))
s=d.snapshot();assert s['channels'][5]['resistance_ohms'] is None and s['channels'][5]['current_mA'] is None
assert s['meter']['current_mA'] is None
assert u.dac.calls==[]
d.submit(dict(action='set',unit=2,channel=6,voltage=.25,request_id='voltage-mode'))
queued=u.commands.get_nowait();assert queued['voltage']==.25 and queued['channel']==2
u.action(queued);u.complete(queued)
assert u.dac.calls[0]==('INIT',2)
d.close();temporary.cleanup()
print('PASS: removing R restores voltage mode without output writes; explicit voltage commands still work.')

## I10 — Dashboard JavaScript interaction checks

The real dashboard script runs against a small DOM double. The test captures button payloads without HTTP requests; it checks labels, converted inputs, meter/history rendering and stale-state disabling. This verifies behavior, not browser layout.

In [ ]:
import subprocess
script=r"""
const fs=require('fs'),vm=require('vm'),assert=require('node:assert/strict');
const elements=new Map();
class Element{
 constructor(){this.value='';this.children=[];this.attrs={};this.hidden=false;this.disabled=false;this.validation='';}
 set innerHTML(value){this.html=value;for(const m of value.matchAll(/<[^>]+\bid="([^"]+)"[^>]*>/g)){const el=new Element();for(const a of m[0].matchAll(/([\w-]+)="([^"]*)"/g))el[a[1]]=a[2];elements.set(m[1],el);}}
 get innerHTML(){return this.html||'';}
 append(...children){this.children.push(...children);}
 replaceChildren(...children){this.children=children;}
 setAttribute(k,v){this.attrs[k]=v;}
 removeAttribute(k){delete this[k];}
 setCustomValidity(value){this.validation=value;}
 reportValidity(){return !this.validation&&this.value!==''&&Number.isFinite(Number(this.value))&&(this.min==null||Number(this.value)>=Number(this.min))&&(this.max==null||Number(this.value)<=Number(this.max));}
}
const body=new Element();body.innerHTML=fs.readFileSync('malab/dashboard/static/index.html','utf8');
const context=vm.createContext({console,performance,AbortSignal,crypto:require('node:crypto').webcrypto,
 document:{getElementById:id=>elements.get(id)||null,createElement:()=>new Element(),createTextNode:text=>({text}),querySelector:()=>({content:'offline-token'})},
 setInterval:()=>1,setTimeout:()=>1,fetch:()=>new Promise(()=>{}),location:{reload:()=>{throw Error('Unexpected reload');}},captures:[]});
vm.runInContext(fs.readFileSync('malab/dashboard/static/app.js','utf8'),context);
const now=Date.now()/1000;
context.fixture={now,server_started:1,channel_count:4,units:[{unit:1,address:'192.0.2.10',first_channel:1,last_channel:4,usb_port:null,network:{state:'good',last_ok:now,latency_ms:1},recovery:{state:'unsupported'},ramp:{state:'IDLE'},pending:null,device_error:null}],
 channels:Array.from({length:4},(_,i)=>({channel:i+1,local_channel:i+1,unit:1,voltage:1,at:now,source:'readback',limits:[-10,10],calibrated:false,resistance_ohms:null,current_mA:null,current_limits_mA:null})),
 meter:{channel:2,address:'192.0.2.71',state:'good',at:now,voltage:1.01,resistance_ohms:null,current_mA:null},jobs:{},events:[],history:[{at:now,channels:[1,1,1,1],currents:[null,.5,null,null],resistances:[null,2000,null,null],meter:1.01,meter_current:.505}]};
const run=code=>vm.runInContext(code,context),el=id=>elements.get(id);
run('state=fixture;received=performance.now();build();render();drawTrace();send=(unit,data)=>captures.push(data);');
assert.equal(el('set-2').textContent,'Set voltage');assert.equal(el('secondary-2').hidden,true);
el('target-2').value='1';el('speed-2').value='.1';
run('Object.assign(state.channels[1],{resistance_ohms:2000,current_mA:.5,current_limits_mA:[-5,5]});Object.assign(state.meter,{resistance_ohms:2000,current_mA:.505});render();drawTrace();');
assert.equal(el('set-2').textContent,'Set current');assert.equal(el('target-label-2').textContent,'Target · mA');assert.equal(el('speed-label-2').textContent,'Ramp · mA/s');
assert.equal(Number(el('target-2').value),.5);assert.equal(Number(el('speed-2').value),.05);assert.equal(Number(el('target-2').max),5);
assert.match(el('voltage-2').innerHTML,/\+0\.50000.*mA/);assert.match(el('secondary-2').textContent,/\+1\.00000 V/);
assert.match(el('meter-voltage').innerHTML,/\+0\.50500.*mA/);assert.match(el('meter-secondary').textContent,/1\.01000 V measured/);
assert.equal(el('trace-mode').value,'current');assert.match(el('trace-note').textContent,/mA/);assert.match(el('trace').innerHTML,/<path/);
el('set-2').onclick();assert.equal(context.captures.at(-1).action,'set_current');assert.equal(context.captures.at(-1).current_mA,.5);assert.equal(context.captures.at(-1).resistance_ohms,2000);
el('form-2').onsubmit({preventDefault(){}});assert.equal(context.captures.at(-1).action,'ramp_current');assert.equal(context.captures.at(-1).speed_mA_s,.05);
el('resistor-2').value='3000';el('save-resistor-2').onclick();assert.equal(context.captures.at(-1).action,'resistor');assert.equal(context.captures.at(-1).resistance_ohms,3000);
const count=context.captures.length;el('resistor-2').value='0';el('save-resistor-2').onclick();assert.equal(context.captures.length,count);
el('remove-resistor-2').onclick();assert.equal(context.captures.at(-1).resistance_ohms,null);
run('state.units[0].network.state="disconnected";render();');assert.equal(el('set-2').disabled,true);
run('state.units[0].network.state="good";received=performance.now()-3000;render();');assert.equal(el('set-2').disabled,true);assert.equal(el('save-resistor-2').disabled,true);
run('received=performance.now();Object.assign(state.channels[1],{resistance_ohms:null,current_mA:null,current_limits_mA:null});Object.assign(state.meter,{resistance_ohms:null,current_mA:null});render();drawTrace();');
assert.equal(el('set-2').textContent,'Set voltage');assert.equal(el('secondary-2').hidden,true);assert.equal(Number(el('target-2').value),1);assert.equal(Number(el('speed-2').value),.1);assert.equal(el('trace-mode').value,'voltage');
el('set-2').onclick();assert.equal(context.captures.at(-1).action,'set');assert.equal(context.captures.at(-1).voltage,1);
console.log('PASS: actual dashboard JS builds cards, switches mA/V labels and targets, emits correct actions, renders meter/history and disables stale controls (DOM double; no browser or live requests).');
"""
result=subprocess.run(["node","-e",script],capture_output=True,text=True,encoding="utf-8")
print(result.stdout)
assert result.returncode==0,result.stderr